[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/topic-jev-vs-llm.ipynb)

# 특강 실습: 결정 모델을 손으로 만들어 보기

**목표.** API 키 없이, 작은 한국어 언어모형의 **다음 토큰 확률만** 가지고 결정 모델의 세 가지 질문 유형(Choice, Score, Noul)을 직접 만든다. 그다음 제주 관광 문의 24건에 대해 **확신도 구간별 정확도**를 재서, 보정(calibration)이 말이 아니라 측정할 수 있는 성질이라는 것을 확인한다.

강의 노트의 확신도 식과 임계값 라우팅을 그대로 코드로 옮긴다. 4주차 노트북에서 쓴 `label_scores()` 와 같은 뿌리다.

## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다. GPU는 필요 없다.

In [ ]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install transformers torch

## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.

### 1-1. 채점셋 - 제주 관광 문의 24건

정답(라벨)이 있는 평가 데이터다. 확신도가 쓸모 있는지 재려면 **정답지가 있어야** 한다. 라벨은 주차, 시설, 요금 세 가지다.

In [ ]:
# 제주 관광 문의 채점셋 (라벨: 주차 / 시설 / 요금)
eval_set = [
    ("성산일출봉 주차장이 어디예요?", "주차"),
    ("함덕해수욕장에 주차할 곳이 있나요?", "주차"),
    ("제주공항에서 렌터카를 어디서 반납하나요?", "주차"),
    ("만장굴 주차 요금이 있나요?", "주차"),
    ("협재해수욕장 주차장이 만차인지 알 수 있나요?", "주차"),
    ("한라산 어리목 탐방로 주차가 가능한가요?", "주차"),
    ("천지연폭포 주차장에서 입구까지 멀어요?", "주차"),
    ("섭지코지 주차 공간이 넓은가요?", "주차"),
    ("이호테우해수욕장에 샤워실이 있나요?", "시설"),
    ("성산일출봉에 화장실이 많이 있나요?", "시설"),
    ("함덕해수욕장에 파라솔을 빌릴 수 있나요?", "시설"),
    ("제주민속촌에 수유실이 있나요?", "시설"),
    ("한라산 국립공원에 매점이 있나요?", "시설"),
    ("월정리해수욕장에 짐 보관함이 있나요?", "시설"),
    ("만장굴 안을 휠체어가 다닐 수 있나요?", "시설"),
    ("식물원에 유모차 대여가 되나요?", "시설"),
    ("성산일출봉 입장료가 얼마예요?", "요금"),
    ("만장굴 입장료 할인이 있나요?", "요금"),
    ("제주민속촌 가족권 가격이 어떻게 되나요?", "요금"),
    ("식물원 입장권을 온라인으로 사면 더 싼가요?", "요금"),
    ("우도 왕복 배 삯이 얼마인가요?", "요금"),
    ("한라산 트레킹은 무료인가요?", "요금"),
    ("청소년은 입장료가 할인되나요?", "요금"),
    ("오름 이용 요금이 따로 있나요?", "요금"),
]

LABELS = ["주차", "시설", "요금"]
print(f"채점셋: {len(eval_set)}건")
for lab in LABELS:
    print(f"  {lab}: {sum(1 for _, y in eval_set if y == lab)}건")

### 1-2. 모델 로딩 - 가중치는 그대로

4주차와 같은 작은 한국어 GPT 모델을 쓴다. 학습시키지 않는다. 다운로드한 가중치 그대로다.

진짜 Jev는 이런 구조가 아니다. 우리는 **일반 LLM의 다음 토큰 확률을 빌려 결정 모델을 흉내 내는 것**이고, 그 차이가 이 실습의 관찰 대상이다.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B-Base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

def next_token_logprobs(prompt_text):
    # 프롬프트 바로 다음 토큰 자리의 로그확률 분포를 통째로 돌려준다
    ids = tokenizer(prompt_text, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(ids).logits
    return torch.log_softmax(logits[0, -1], dim=-1)

def first_token_id(text):
    return tokenizer(text, add_special_tokens=False).input_ids[0]

print("모델 준비 완료. 갱신하는 파라미터: 0")

### 1-3. 확신도 계산 - 강의 노트의 식 그대로

후보가 k개일 때 강의 노트에 나온 식은 이것이다.

```text
confidence = (k x 가장 큰 확률 - 1) / (k - 1)
```

k에 3을 넣으면 `(3 x p - 1) / 2` 가 된다. 모든 후보가 똑같으면 0, 한 후보가 1.0이면 1이 나오는지 아래에서 확인한다.

In [ ]:
def confidence_from(probs):
    # probs: 후보별 확률 리스트 (합이 1)
    k = len(probs)
    if k < 2:
        return 1.0
    return (k * max(probs) - 1) / (k - 1)

# 강의 노트의 두 예를 그대로 넣어 본다
print("주차 0.62 / 요금 0.28 / 시설 0.10 ->", round(confidence_from([0.62, 0.28, 0.10]), 2))
print("요금 0.94 / 주차 0.04 / 시설 0.02 ->", round(confidence_from([0.94, 0.04, 0.02]), 2))
print("완전히 균등 (1/3씩)        ->", round(confidence_from([1/3, 1/3, 1/3]), 2))
print("한 곳에 몰빵 (1.0)          ->", round(confidence_from([1.0, 0.0, 0.0]), 2))

### 1-4. Choice - 목록에서 하나 고르기

후보의 첫 토큰끼리만 로그확률을 비교하고, 그 셋만으로 다시 softmax를 걸어 확률로 만든다. **목록 밖의 답은 나올 수 없다.** 후보 셋만 쳐다보기 때문이다.

이것이 결정 모델의 스키마 보장을 손으로 흉내 낸 것이다.

> 한 가지 단순화를 짚고 간다. 주차, 시설, 요금은 이 토크나이저에서 토큰 두 개로 쪼개진다. 우리는 **첫 토큰만** 비교한다. 첫 토큰이 서로 다르므로 구분에는 지장이 없지만, 진짜 결정 모델처럼 후보 전체를 평가하는 것은 아니다.

In [ ]:
CHOICE_PROMPT = """아래 제주 관광 문의를 주차, 시설, 요금 중 하나로 분류하시오.
라벨만 답하시오.

문의: 함덕해수욕장 주차장이 어디예요?
라벨: 주차

문의: 오름 입장료 할인이 있나요?
라벨: 요금

문의: {state}
라벨:"""

def choice(state, options=LABELS, template=CHOICE_PROMPT):
    logps = next_token_logprobs(template.format(state=state))
    raw = torch.tensor([logps[first_token_id(" " + o)] for o in options])
    probs = torch.softmax(raw, dim=-1).tolist()   # 후보 셋에 대해서만 정규화
    best = max(range(len(options)), key=lambda i: probs[i])
    return {
        "choice": options[best],
        "probabilities": {o: round(p, 4) for o, p in zip(options, probs)},
        "confidence": confidence_from(probs),
    }

ans = choice("성산일출봉 주차장 입구가 막혀서 30분째 못 들어가고 있습니다.")
print("choice      :", ans["choice"])
print("probabilities:", ans["probabilities"])
print("confidence  :", round(ans["confidence"], 3))

### 1-5. Noul - 이 문장이 참인가

예/아니오 두 후보만 놓고 같은 계산을 한다. 돌려주는 값은 **참일 확률 하나**다. 답과 확신도가 한 숫자에 합쳐져 있다.

In [ ]:
NOUL_PROMPT = """아래 제주 관광 문의를 읽고 질문에 예 또는 아니오로만 답하시오.

문의: {state}
질문: {question}
답:"""

def noul(state, question):
    logps = next_token_logprobs(NOUL_PROMPT.format(state=state, question=question))
    raw = torch.tensor([logps[first_token_id(" 예")], logps[first_token_id(" 아니오")]])
    probs = torch.softmax(raw, dim=-1).tolist()
    return probs[0]   # 참일 확률

ticket = "성산일출봉 주차장 입구가 막혀서 30분째 못 들어가고 있습니다. 예약한 입장권 환불도 알아보고 싶어요."
print("긴급     :", round(noul(ticket, "지금 바로 대응해야 하는 상황인가?"), 3))
print("환불요청 :", round(noul(ticket, "환불이나 요금 반환을 요구하고 있는가?"), 3))

### 1-6. Score - 순서가 있는 눈금 위의 한 점

단계마다 설명을 붙여 주고, 모델에게 단계 번호를 답하게 한다. 돌려주는 값은 정수가 아니라 **단계별 확률로 가중평균한 값**이다.

```text
score = 0 x p0 + 1 x p1 + 2 x p2
```

돌려주는 confidence도 같이 본다. **이 값이 낮게 나올 것이다.** 0.6B짜리 기본 모델에게 3단계 루브릭은 버거운 과제이고, 낮은 확신도는 그 사실을 숨기지 않고 알려 준다. 값이 낮다고 고장난 것이 아니라, 못 가리겠다는 말을 정직하게 하고 있는 것이다.

In [ ]:
SCORE_PROMPT = """아래 제주 관광 문의가 얼마나 급한지 단계 번호로만 답하시오.

0단계: 가벼운 문의. 지금 불편하지 않다
1단계: 불편하지만 우회할 방법이 있다
2단계: 지금 현장에서 막혀 있다

문의: {state}
단계: """

LEVELS = [
    "가벼운 문의. 지금 불편하지 않다",
    "불편하지만 우회할 방법이 있다",
    "지금 현장에서 막혀 있다",
]

def score(state, levels=LEVELS):
    logps = next_token_logprobs(SCORE_PROMPT.format(state=state))
    raw = torch.tensor([logps[first_token_id(str(i))] for i in range(len(levels))])
    probs = torch.softmax(raw, dim=-1).tolist()
    return {
        "score": sum(i * p for i, p in enumerate(probs)),
        "probabilities": {i: round(p, 4) for i, p in enumerate(probs)},
        "legend": {i: t for i, t in enumerate(levels)},
        "confidence": confidence_from(probs),
    }

s = score(ticket)
print("score      :", round(s["score"], 2))
print("probabilities:", s["probabilities"])
print("confidence :", round(s["confidence"], 3))

### 1-7. fan-out - 한 상태에 네 질문을 한꺼번에

진짜 Jev는 이 넷을 **한 번의 호출로 병렬 평가**한다. 우리 흉내는 네 번 따로 계산하므로 그만큼 느리다. 이 차이가 강의 노트에서 본 지연 시간 차이의 뿌리다.

쓰는 문의는 강의 노트의 그 문의다. **주차 이야기로 시작해 환불로 끝난다.** 우리 흉내가 무엇을 골랐는지, 확신도가 얼마인지 보자. 강의 노트에서 본 판단 오류가 실제로 여기서 나올 수 있다.

In [ ]:
import time

def fan_out(state):
    t0 = time.time()
    result = {
        "분류": choice(state),
        "긴급": noul(state, "지금 바로 대응해야 하는 상황인가?"),
        "심각도": score(state),
        "환불요청": noul(state, "환불이나 요금 반환을 요구하고 있는가?"),
    }
    result["_걸린시간"] = round(time.time() - t0, 2)
    return result

r = fan_out(ticket)
print(f'분류      : {r["분류"]["choice"]}  (confidence {r["분류"]["confidence"]:.2f})')
print(f'긴급      : {r["긴급"]:.2f}')
print(f'심각도    : {r["심각도"]["score"]:.2f}')
print(f'환불요청  : {r["환불요청"]:.2f}')
print(f'네 질문에 걸린 시간: {r["_걸린시간"]}초 (따로 네 번 계산한 결과다)')

## 2. 확신도가 정말 쓸모 있는가

여기가 이 노트북의 핵심이다. 확신도는 **높을 때 실제로 더 자주 맞아야** 쓸모가 있다. 그것을 재는 방법은 간단하다.

1. 채점셋 24건을 전부 분류한다
2. 확신도 구간별로 나눈다
3. 구간마다 정확도를 센다

확신도가 높은 구간의 정확도가 실제로 더 높으면 보정이 된 것이고, 아니면 아닌 것이다. **믿을 말이 아니라 재 볼 값이다.**

In [ ]:
def run_eval(threshold_low, threshold_high):
    rows = []
    for sentence, gold in eval_set:
        a = choice(sentence)
        rows.append((sentence, gold, a["choice"], a["confidence"]))

    buckets = {
        f"낮음 (< {threshold_low})": [],
        f"중간 ({threshold_low} ~ {threshold_high})": [],
        f"높음 (> {threshold_high})": [],
    }
    names = list(buckets)
    for row in rows:
        conf = row[3]
        if conf < threshold_low:
            buckets[names[0]].append(row)
        elif conf < threshold_high:
            buckets[names[1]].append(row)
        else:
            buckets[names[2]].append(row)

    overall = sum(1 for r in rows if r[1] == r[2]) / len(rows)
    print(f"전체 정확도: {overall:.2f}  ({len(rows)}건)")
    print()
    print(f"{'구간':<22} {'건수':>4} {'정확도':>8}")
    for name in names:
        b = buckets[name]
        if not b:
            print(f"{name:<22} {0:>4} {'-':>8}")
            continue
        acc = sum(1 for r in b if r[1] == r[2]) / len(b)
        print(f"{name:<22} {len(b):>4} {acc:>8.2f}")
    return rows, buckets

rows, buckets = run_eval(0.60, 0.85)

### 2-1. 틀린 건들이 어느 구간에 모여 있나

틀린 건만 뽑아 확신도 순으로 본다. **확신도가 낮은 쪽에 몰려 있으면** 임계값으로 걸러 낼 수 있다는 뜻이다. 높은 쪽에 섞여 있으면 그 임계값은 위험하다.

In [ ]:
wrong = sorted([r for r in rows if r[1] != r[2]], key=lambda r: -r[3])
print(f"틀린 건: {len(wrong)}건 / {len(rows)}건")
print()
print(f"{'확신도':>6}  {'정답':<4} {'예측':<4} 문의")
for sentence, gold, pred, conf in wrong:
    print(f"{conf:>6.2f}  {gold:<4} {pred:<4} {sentence}")

### 2-2. 임계값을 적용하면 무엇이 자동화되나

강의 노트의 라우팅 코드를 그대로 돌려 본다. 자동 처리로 넘어간 건들의 정확도가 전체 정확도보다 **높아야** 자동화에 의미가 있다.

In [ ]:
def route(rows, threshold_low, threshold_high):
    auto = [r for r in rows if r[3] > threshold_high]
    confirm = [r for r in rows if threshold_low < r[3] <= threshold_high]
    human = [r for r in rows if r[3] <= threshold_low]

    def acc(b):
        return f"{sum(1 for r in b if r[1] == r[2]) / len(b):.2f}" if b else "-"

    print(f"자동 처리      : {len(auto):>2}건, 정확도 {acc(auto)}")
    print(f"사용자에게 확인 : {len(confirm):>2}건, 정확도 {acc(confirm)}")
    print(f"사람에게 넘김   : {len(human):>2}건, 정확도 {acc(human)}")
    print()
    if auto:
        missed = sum(1 for r in auto if r[1] != r[2])
        print(f"-> 사람 확인 없이 나간 {len(auto)}건 중 {missed}건이 틀렸다")
    else:
        print("-> 자동 처리로 넘어간 건이 하나도 없다. 임계값이 이 모델에 비해 높다는 뜻이다")

route(rows, 0.60, 0.85)

## 3. 한 지점만 바꿔 보기 - 임계값

아래 셀의 `# TODO` 로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 규칙: 두 임계값을 바꿔 보고, 자동 처리 건수와 그 구간의 정확도가 어떻게 움직이는지 본다. 바꾸기 전 숫자를 적어 두면 비교할 수 있습니다.
> 임계값을 낮추면 자동 처리는 늘고 그 안의 오류도 늘어난다. 어디까지 허용할지가 설계 판단이다.

In [ ]:
# TODO: 아래 두 임계값만 바꾼다. (0 과 1 사이, LOW < HIGH)
THRESHOLD_LOW = 0.60
THRESHOLD_HIGH = 0.85

# 아래는 그대로 둡니다
rows2, _ = run_eval(THRESHOLD_LOW, THRESHOLD_HIGH)
print()
route(rows2, THRESHOLD_LOW, THRESHOLD_HIGH)

## 4. 확인 질문

1. 확신도가 높은 구간의 정확도가 낮은 구간보다 실제로 높았나요? 그렇지 않았다면, 이 작은 모델의 확신도에 대해 무엇을 말해 주는 결과인가요?
2. 틀린 건들의 확신도는 대체로 높았나요, 낮았나요? 확신도로 걸러 낼 수 있는 오류였나요?
3. 임계값을 낮추면 자동 처리 건수와 그 구간의 오류 수가 각각 어떻게 움직였나요? 둘을 동시에 좋게 만들 수 있었나요?
4. 이 노트북의 `choice()` 는 목록 밖의 답을 절대 내놓을 수 없습니다. 그런데도 틀린 건이 남았습니다. 강의 노트의 어느 구분에 해당하나요?

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.

*(여기에 답을 적으세요)*

## 5. (참고) 진짜 결정 모델은 어떻게 부르나

아래는 실제 Jev API를 쓰는 코드다. **유료 키가 필요하므로 이 노트북에서는 실행하지 않는다.** 우리가 위에서 손으로 만든 세 함수가 실제 API에서 어떤 모양인지 비교해 보는 용도다.

차이를 세 가지만 짚어 보자.

- 네 질문이 `questions` 하나에 묶여 **한 번의 호출**로 나간다 (우리는 네 번 따로 계산했다)
- 후보에 설명(`criteria`)을 붙일 수 있다 (우리는 라벨 이름만 썼다)
- 돌아오는 확률이 **보정되도록 훈련**돼 있다 (우리 확률은 그런 보장이 없다)

In [ ]:
# 실행하지 않습니다. API 키가 필요합니다.
#
# !pip install typesafe-sdk
# import os
# os.environ["TYPESAFE_API_KEY"] = "..."
#
# from typesafe_sdk import Choice, Noul, Score, TypeSafeClient
#
# with TypeSafeClient() as client:
#     response = client.system_one(
#         model="jev-latest",
#         state=ticket,
#         questions={
#             "분류": Choice(
#                 instructions="이 문의를 어느 갈래로 보내야 하는가?",
#                 criteria={
#                     "주차": "주차장 위치, 만차, 진입",
#                     "시설": "화장실, 매점, 편의시설",
#                     "요금": "입장료, 할인, 환불",
#                 },
#             ),
#             "긴급": Noul(instructions="지금 바로 대응해야 하는 상황인가?"),
#             "심각도": Score(instructions="불편이 얼마나 큰가?", criteria=LEVELS),
#         },
#     )
#
# print(response.answers["분류"].choice, response.answers["분류"].confidence)
# print(response.answers["긴급"].noul)
# print(response.answers["심각도"].score)

print("이 셀은 참고용입니다. 위 코드는 주석 처리돼 있어 아무것도 실행되지 않습니다.")

## 6. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/deepnlp-2026)의 `assignments/week-04/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.